# Luz Polarizada y Fase Geométrica en Unidades qg: Óptica Cuántica y Redes QPON

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Viny2030/qang/blob/main/notebooks/qang_optics_es.ipynb)

Este cuaderno implementa de manera rigurosa la unificación entre polarimetría de Stokes, matrices de Mueller y la métrica $qg$, tal como se sistematiza en la obra reescrita *Computación, información y física cuántica en el espacio qang* (35 capítulos, Parte VII y aplicaciones a redes ópticas cuánticas QPON).

### Ejes de la Formulación Óptica en qang (v0.14.0):
1. **Parámetros de Stokes como valores $qg$:** La esfera de Poincaré equivale a la esfera de Bloch. Con $|H\rangle = |0\rangle$ y $|V\rangle = |1\rangle$:
   * $qg_Z = S_1 / S_0$ ($H - V$)
   * $qg_X = S_2 / S_0$ ($D - A$)
   * $qg_Y = S_3 / S_0$ ($R - L$)
2. **Matrices de Mueller sin pérdidas:** Equivalen estrictamente a la regla de compuertas cuánticas $qg'_P = qg_{U^\dagger P U}$.
3. **Ley de Malus y Polarimetría Estadística:** Proyección directa mediante conteo de fotones y estimación de intervalos de confianza en escala $mqg$.
4. **Fase Geométrica de Berry y Pancharatnam:** Correspondencia con el teorema de Stokes sobre la esfera: $\gamma = -\Omega / 2$, i.e. **$(qg_Z - 1)/2$ vueltas**.

## 1. Instalación y Carga de Módulos (`qang 0.14.0`)

In [ ]:
import importlib, subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "qang>=0.14.0", "matplotlib", "numpy"], check=False)
importlib.invalidate_caches()

import numpy as np
import matplotlib.pyplot as plt
import qang
from qang import formulation as F
from qang import geometric as G
from qang import polarization as P

print(f"Librería qang inicializada: v{qang.__version__}")

## 2. Los Parámetros de Stokes son Valores $qg$ Exactos
Se comprueban estados fundamentales (Horizontal, Vertical, Diagonal, Antidiagonal, Circular Derecho, Circular Izquierdo y Elíptico) contrastando `qang.polarization` con los valores de expectación de Pauli de `qang.formulation`.

In [ ]:
s2 = np.sqrt(2)
states = {
    "H": [1, 0], 
    "V": [0, 1], 
    "D": [1 / s2, 1 / s2], 
    "A": [1 / s2, -1 / s2],
    "R": [1 / s2, 1j / s2], 
    "L": [1 / s2, -1j / s2], 
    "Elíptico": [0.6, 0.8 * np.exp(0.7j)]
}

print(f"{'Estado':<12}{'qang (Stokes -> qg)':>35}{'Estándar (<P>)':>35}{'Diferencia':>14}")
fmt = lambda d: "(" + ", ".join(f"{d[k]:+.3f}" for k in "XYZ") + ")"
for name, E in states.items():
    q = P.jones_to_qg(E)
    ref = F.qg_values(np.array(E, dtype=complex))
    ref = {k: ref.get(k, 0.0) for k in "XYZ"}
    diff = max(abs(q[k] - ref[k]) for k in "XYZ")
    print(f"{name:<12}{fmt(q):>35}{fmt(ref):>35}{diff:>14.1e}")

## 3. Luz Parcialmente Polarizada y Estados Mezcla
El grado de polarización $P$ corresponde a la norma del vector $qg$, y la pureza del estado se determina mediante $\mathrm{Tr}(\rho^2) = \frac{1 + P^2}{2}$.

In [ ]:
E = np.array([0.6, 0.8 * np.exp(0.7j)])
print("Despolarización | Grado Pol. P | Pureza (qang) | Tr(rho^2)")
for p in (0.0, 0.3, 0.6, 0.9):
    S = P.depolarizer(1 - p) @ P.jones_to_stokes(E)
    q = P.stokes_to_qg(S)
    rho = F.state_from_qg({"I": 1.0, **q}, 1)
    deg_p = P.degree_of_polarization(S)
    purity_qg = P.purity_from_stokes(S)
    tr_rho2 = np.real(np.trace(rho @ rho))
    print(f"     {p:.1f}        |    {deg_p:.3f}     |     {purity_qg:.4f}    |   {tr_rho2:.4f}")

## 4. Matriz de Mueller sin Pérdidas como Regla de Compuertas $qg$
Para placas de onda y rotadores ópticos, la acción óptica de Mueller sobre el vector de Stokes coincide numéricamente con $qg'_P = qg_{U^\dagger P U}$.

In [ ]:
rng = np.random.default_rng(42)
components = {
    "Media onda 0.3 rad": P.half_wave_plate(0.3),
    "Cuarto de onda 1.1 rad": P.quarter_wave_plate(1.1),
    "Rotador 0.7 rad": P.rotator(0.7)
}

for name, J in components.items():
    max_err = 0.0
    for _ in range(10):
        E = rng.normal(size=2) + 1j * rng.normal(size=2)
        E /= np.linalg.norm(E)
        q_in = P.jones_to_qg(E)
        q_mueller = P.stokes_to_qg(P.mueller_from_jones(J) @ P.qg_to_stokes(q_in))
        q_rule = F.apply_gate({"I": 1.0, **q_in}, J)
        max_err = max(max_err, max(abs(q_mueller[k] - q_rule.get(k, 0.0)) for k in "XYZ"))
    print(f"{name:<24} Máx discrepancia Mueller vs regla qg: {max_err:.1e}")

## 5. Ley de Malus en Coordenadas $qg$ y Redes Cuánticas
La intensidad transmitida por un polarizador a ángulo $\theta$ obedece:
$$I = I_0 \frac{1 + qg_Z \cos 2\theta + qg_X \sin 2\theta}{2}$$
independientemente del estado de entrada.

In [ ]:
theta = np.linspace(0, np.pi, 200)
fig, ax = plt.subplots(figsize=(6.5, 3.4))

plot_states = [("H", [1, 0], "#2a78d6"), ("D", [1/s2, 1/s2], "#eb6834"), 
               ("R", [1/s2, 1j/s2], "#1baf7a"), ("Elíptico", [0.6, 0.8 * np.exp(0.7j)], "#5f5e58")]

for name, E, color in plot_states:
    q = P.jones_to_qg(E)
    ax.plot(theta, [P.malus_intensity(q, t) for t in theta], color=color, lw=1.8, label=name)
    pts = np.linspace(0, np.pi, 9)
    ax.plot(pts, [(P.mueller_from_jones(P.linear_polarizer(t)) @ P.qg_to_stokes(q))[0] for t in pts], "o", color=color, ms=4)

ax.set_xlabel("Ángulo del polarizador θ (rad)")
ax.set_ylabel("Intensidad normalizada I/I₀")
ax.set_title("Ley de Malus en unidades qg (Líneas: qg, Puntos: Mueller)")
ax.legend(frameon=False, ncol=4, fontsize=8, loc="upper center")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Fase Geométrica y Teorema de Stokes
Un lazo a latitud constante $qg_Z$ encierra un ángulo sólido $\Omega = 2\pi(1 - qg_Z)$. La fase geométrica resultante es exactamente:
$$\gamma = -\frac{\Omega}{2} \implies \frac{qg_Z - 1}{2} \text{ vueltas}$$
Se verifica mediante tres metodologías independientes:
1. Solapamientos sucesivos de estados (Pancharatnam).
2. Cálculo analítico del ángulo sólido $\Omega$.
3. Integración de flujo de curvatura de Berry.

In [ ]:
def generate_cone(qg_z, n=2000):
    s = np.sqrt(1 - qg_z**2)
    t = np.linspace(0, 2 * np.pi, n, endpoint=False)
    return np.stack([s * np.cos(t), s * np.sin(t), np.full(n, qg_z)], axis=1)

zs = np.linspace(-0.9, 0.9, 7)
over = [G.geometric_phase_qg(generate_cone(z)).phi for z in zs]
solid = [(G.berry_phase_from_solid_angle(G.solid_angle(generate_cone(z), reference=[0, 0, 1])) / (2 * np.pi)) % 1 for z in zs]
flux = [(-G.curvature_flux(z) / (2 * np.pi)) % 1 for z in zs]

zz = np.linspace(-1, 0.995, 200)
fig, ax = plt.subplots(figsize=(6.5, 3.4))
ax.plot(zz, ((zz - 1) / 2) % 1, color="#5f5e58", lw=2, label="Fórmula (qg_Z - 1)/2 mod 1")
ax.plot(zs, over, "o", color="#2a78d6", ms=7, label="Solapamientos (Pancharatnam)")
ax.plot(zs, solid, "s", color="#eb6834", ms=5, label="Ángulo Sólido")
ax.plot(zs, flux, "^", color="#1baf7a", ms=4, label="Flujo de Curvatura (Stokes)")

ax.set_xlabel("qg_Z del lazo cónico")
ax.set_ylabel("Fase Geométrica (vueltas)")
ax.set_title("Teorema de Stokes y Fase de Berry en Métrica qg")
ax.legend(frameon=False, fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

max_diff = max(abs(((a - (z - 1) / 2) + 0.5) % 1 - 0.5) for a, z in zip(over, zs))
print(f"Máxima diferencia numérica con la predicción analítica: {max_diff:.2e}")

## Conclusiones
* **Unificación Óptica-Cuántica:** Se establece la equivalencia formal entre polarimetría clásica de Stokes y la unidad angular-probabilística $qg$.
* **Diagnóstico de Redes Ópticas (QPON):** La métrica permite auditar compuertas fotónicas lineales e interferómetros sin depender de parametrizaciones trigonométricas ad-hoc.
* **Consistencia:** Alineado al libro principal de 35 capítulos y validado por los módulos de `qang 0.14.0`.